# UAVDT Notebook 04 — Metric BEV Calibration and Dynamic 3D Scene Export

This notebook continues after Notebook 03. It converts BEV pixel coordinates into approximate metric coordinates, builds physically-sized 3D vehicle cuboids, and exports the dynamic scene for later rendering/simulation.

Inputs from previous notebooks:
- `notebook_02_bev_homography/homography_image_to_bev.json`
- `notebook_03_bev_vehicles/vehicle_tracks_bev.csv`

Outputs:
- metric calibration JSON
- metric vehicle tracks CSV
- per-frame 3D cuboids CSV
- simple OBJ file for one frame
- dynamic scene JSON

The calibration is approximate. For this UAV traffic setup, the most practical automatic scale priors are lane width or average vehicle size.


In [ ]:
#@title 1. Set local project paths
from pathlib import Path
import json
import math
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from notebook_local import resolve_project_dir, print_local_setup

PROJECT_DIR = resolve_project_dir()
WORK_DIR = PROJECT_DIR / 'work'
NB02_DIR = WORK_DIR / 'notebook_02_bev_homography'
NB03_DIR = WORK_DIR / 'notebook_03_bev_vehicles'
NB04_DIR = WORK_DIR / 'notebook_04_metric_scene_export'
NB04_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('Notebook 02 directory:', NB02_DIR)
print('Notebook 03 directory:', NB03_DIR)
print('Notebook 04 output directory:', NB04_DIR)

assert NB02_DIR.exists(), 'Notebook 02 output directory not found. Run Notebook 02 first.'
assert NB03_DIR.exists(), 'Notebook 03 output directory not found. Run Notebook 03 first.'


In [ ]:
#@title 2. Load homography and vehicle BEV tracks
homography_candidates = [
    NB02_DIR / 'homography_image_to_bev.json',
    NB02_DIR / 'bev_homography.json',
    NB02_DIR / 'homography.json',
]

homography_path = None
for p in homography_candidates:
    if p.exists():
        homography_path = p
        break

if homography_path is None:
    found = sorted(NB02_DIR.glob('*.json'))
    print('JSON files found in Notebook 02 directory:')
    for p in found:
        print(' ', p)
    raise FileNotFoundError('Could not find a homography JSON file from Notebook 02.')

with open(homography_path, 'r') as f:
    homography_data = json.load(f)

print('Loaded homography JSON:', homography_path)
print('Homography keys:', list(homography_data.keys()))

BEV_WIDTH = int(homography_data.get('bev_width', homography_data.get('BEV_WIDTH', 600)))
BEV_HEIGHT = int(homography_data.get('bev_height', homography_data.get('BEV_HEIGHT', 1000)))
print('BEV size:', BEV_WIDTH, 'x', BEV_HEIGHT)

track_candidates = [
    NB03_DIR / 'vehicle_tracks_bev.csv',
    NB03_DIR / 'tracks_bev.csv',
    NB03_DIR / 'detections_tracks_bev.csv',
]

tracks_path = None
for p in track_candidates:
    if p.exists():
        tracks_path = p
        break

if tracks_path is None:
    found = sorted(NB03_DIR.glob('*.csv'))
    print('CSV files found in Notebook 03 directory:')
    for p in found:
        print(' ', p)
    raise FileNotFoundError('Could not find vehicle tracks CSV from Notebook 03.')

tracks = pd.read_csv(tracks_path)
print('Loaded tracks CSV:', tracks_path)
print('Rows:', len(tracks))
print('Columns:', list(tracks.columns))
display(tracks.head())

rename_map = {}
for old, new in [
    ('bev_x', 'bev_x_px'),
    ('bev_y', 'bev_y_px'),
    ('x_bev', 'bev_x_px'),
    ('y_bev', 'bev_y_px'),
    ('track', 'track_id'),
    ('id', 'track_id'),
    ('frame', 'frame_id'),
]:
    if old in tracks.columns and new not in tracks.columns:
        rename_map[old] = new
tracks = tracks.rename(columns=rename_map)

required = ['frame_id', 'track_id', 'bev_x_px', 'bev_y_px']
missing = [c for c in required if c not in tracks.columns]
if missing:
    raise ValueError('Missing required columns after normalization: ' + str(missing))

tracks['frame_id'] = tracks['frame_id'].astype(int)
tracks['track_id'] = tracks['track_id'].astype(int)
print('Normalized columns are OK.')


## 3. Choose metric calibration mode

The BEV homography produces coordinates in BEV pixels. To create useful 3D boxes, we need an approximate pixel-to-meter scale.

Supported modes:

1. `auto_vehicle_width_prior`: infer scale from median detected vehicle width, assuming an average car width.
2. `manual_lane_width`: provide two BEV points across one lane and the known lane width.
3. `manual_pixels_per_meter`: directly set pixels-per-meter.

For a fully unmanned first pass, use `auto_vehicle_width_prior`. For better accuracy, use lane width calibration later.


In [ ]:
#@title 3A. Calibration settings
CALIBRATION_MODE = 'auto_vehicle_width_prior' #@param ['auto_vehicle_width_prior', 'manual_lane_width', 'manual_pixels_per_meter']

ASSUMED_CAR_WIDTH_M = 1.8 #@param {type:'number'}

LANE_POINT_A = [250, 520] #@param
LANE_POINT_B = [350, 520] #@param
ASSUMED_LANE_WIDTH_M = 3.5 #@param {type:'number'}

MANUAL_PIXELS_PER_METER = 25.0 #@param {type:'number'}

print('Calibration mode:', CALIBRATION_MODE)


In [ ]:
#@title 3B. Estimate pixels-per-meter
def infer_vehicle_width_px_from_tracks(df):
    possible_width_cols = ['bev_box_width_px', 'box_width_bev_px', 'width_bev_px']
    for c in possible_width_cols:
        if c in df.columns:
            vals = pd.to_numeric(df[c], errors='coerce').dropna()
            vals = vals[(vals > 2) & (vals < 300)]
            if len(vals) >= 3:
                return float(vals.median()), c

    if {'x1', 'x2'}.issubset(df.columns):
        vals = (pd.to_numeric(df['x2'], errors='coerce') - pd.to_numeric(df['x1'], errors='coerce')).dropna().abs()
        vals = vals[(vals > 2) & (vals < 300)]
        if len(vals) >= 3:
            return float(vals.median()), 'image_bbox_width_fallback'

    return 45.0, 'fallback_constant'

if CALIBRATION_MODE == 'auto_vehicle_width_prior':
    median_width_px, source_col = infer_vehicle_width_px_from_tracks(tracks)
    pixels_per_meter = median_width_px / ASSUMED_CAR_WIDTH_M
    calibration_details = {
        'mode': CALIBRATION_MODE,
        'assumed_car_width_m': ASSUMED_CAR_WIDTH_M,
        'median_width_px': median_width_px,
        'width_source': source_col,
    }
elif CALIBRATION_MODE == 'manual_lane_width':
    a = np.array(LANE_POINT_A, dtype=float)
    b = np.array(LANE_POINT_B, dtype=float)
    lane_width_px = float(np.linalg.norm(a - b))
    pixels_per_meter = lane_width_px / ASSUMED_LANE_WIDTH_M
    calibration_details = {
        'mode': CALIBRATION_MODE,
        'lane_point_a': LANE_POINT_A,
        'lane_point_b': LANE_POINT_B,
        'assumed_lane_width_m': ASSUMED_LANE_WIDTH_M,
        'lane_width_px': lane_width_px,
    }
elif CALIBRATION_MODE == 'manual_pixels_per_meter':
    pixels_per_meter = float(MANUAL_PIXELS_PER_METER)
    calibration_details = {
        'mode': CALIBRATION_MODE,
        'manual_pixels_per_meter': pixels_per_meter,
    }
else:
    raise ValueError('Unknown CALIBRATION_MODE: ' + CALIBRATION_MODE)

meters_per_pixel = 1.0 / pixels_per_meter
print('Pixels per meter:', pixels_per_meter)
print('Meters per pixel:', meters_per_pixel)
print('Calibration details:', calibration_details)

calibration_json = {
    'pixels_per_meter': pixels_per_meter,
    'meters_per_pixel': meters_per_pixel,
    'bev_width_px': BEV_WIDTH,
    'bev_height_px': BEV_HEIGHT,
    'details': calibration_details,
}
calibration_path = NB04_DIR / 'metric_calibration.json'
with open(calibration_path, 'w') as f:
    json.dump(calibration_json, f, indent=2)
print('Saved calibration:', calibration_path)


In [ ]:
#@title 4. Convert BEV tracks from pixels to meters
metric = tracks.copy()
metric['x_m'] = metric['bev_x_px'] * meters_per_pixel
metric['y_m'] = (BEV_HEIGHT - metric['bev_y_px']) * meters_per_pixel

FPS = 30.0 #@param {type:'number'}
metric['t_s'] = metric['frame_id'] / FPS

metric = metric.sort_values(['track_id', 'frame_id']).reset_index(drop=True)
metric['vx_mps'] = np.nan
metric['vy_mps'] = np.nan
metric['speed_mps'] = np.nan
metric['yaw_rad'] = np.nan

for track_id, group in metric.groupby('track_id'):
    idx = group.index.to_numpy()
    xs = group['x_m'].to_numpy(dtype=float)
    ys = group['y_m'].to_numpy(dtype=float)
    ts = group['t_s'].to_numpy(dtype=float)

    if len(idx) == 1:
        metric.loc[idx, ['vx_mps', 'vy_mps', 'speed_mps', 'yaw_rad']] = [0.0, 0.0, 0.0, 0.0]
        continue

    vx = np.zeros(len(idx), dtype=float)
    vy = np.zeros(len(idx), dtype=float)
    yaw = np.zeros(len(idx), dtype=float)
    speed = np.zeros(len(idx), dtype=float)

    for i in range(len(idx)):
        if i == 0:
            j0, j1 = 0, 1
        elif i == len(idx) - 1:
            j0, j1 = len(idx) - 2, len(idx) - 1
        else:
            j0, j1 = i - 1, i + 1

        dt = max(ts[j1] - ts[j0], 1e-6)
        dx = xs[j1] - xs[j0]
        dy = ys[j1] - ys[j0]
        vx[i] = dx / dt
        vy[i] = dy / dt
        speed[i] = math.hypot(vx[i], vy[i])
        yaw[i] = math.atan2(vy[i], vx[i])

    metric.loc[idx, 'vx_mps'] = vx
    metric.loc[idx, 'vy_mps'] = vy
    metric.loc[idx, 'speed_mps'] = speed
    metric.loc[idx, 'yaw_rad'] = yaw

metric_path = NB04_DIR / 'vehicle_tracks_metric.csv'
metric.to_csv(metric_path, index=False)
print('Saved metric tracks:', metric_path)
print('Rows:', len(metric))
display(metric.head())


In [ ]:
#@title 5. Build metric 3D vehicle cuboids
DEFAULT_DIMS = {
    'car': (4.5, 1.8, 1.5),
    'truck': (8.0, 2.5, 3.0),
    'bus': (12.0, 2.6, 3.2),
    'motorcycle': (2.2, 0.8, 1.4),
    'bicycle': (1.8, 0.6, 1.4),
    'unknown': (4.5, 1.8, 1.5),
}

def normalize_class_name(row):
    for c in ['class_name', 'class', 'label', 'category']:
        if c in row.index and pd.notna(row[c]):
            name = str(row[c]).lower()
            if 'bus' in name:
                return 'bus'
            if 'truck' in name:
                return 'truck'
            if 'motor' in name:
                return 'motorcycle'
            if 'bike' in name or 'bicycle' in name:
                return 'bicycle'
            if 'car' in name or 'vehicle' in name:
                return 'car'
    return 'car'

cuboids = []
for _, row in metric.iterrows():
    cls = normalize_class_name(row)
    length_m, width_m, height_m = DEFAULT_DIMS.get(cls, DEFAULT_DIMS['unknown'])
    cuboids.append({
        'frame_id': int(row['frame_id']),
        'track_id': int(row['track_id']),
        'class_name': cls,
        'center_x_m': float(row['x_m']),
        'center_y_m': float(row['y_m']),
        'center_z_m': height_m / 2.0,
        'length_m': length_m,
        'width_m': width_m,
        'height_m': height_m,
        'yaw_rad': float(row['yaw_rad']),
        'speed_mps': float(row['speed_mps']),
    })

cuboids_df = pd.DataFrame(cuboids)
cuboids_path = NB04_DIR / 'vehicle_cuboids_metric.csv'
cuboids_df.to_csv(cuboids_path, index=False)
print('Saved metric cuboids:', cuboids_path)
print('Rows:', len(cuboids_df))
display(cuboids_df.head())


In [ ]:
#@title 6. Plot metric BEV tracks
fig_width = 8
fig_height = fig_width * BEV_HEIGHT / BEV_WIDTH
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

for track_id, group in metric.groupby('track_id'):
    group = group.sort_values('frame_id')
    ax.plot(group['x_m'], group['y_m'], marker='o', linewidth=1, markersize=3)
    last = group.iloc[-1]
    ax.text(last['x_m'], last['y_m'], str(track_id), fontsize=8)

ax.set_title('Metric BEV vehicle tracks')
ax.set_xlabel('X meters')
ax.set_ylabel('Y meters')
ax.set_aspect('equal', adjustable='box')
ax.grid(True)

bev_metric_overlay_path = NB04_DIR / 'metric_bev_tracks.png'
plt.savefig(bev_metric_overlay_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved metric BEV tracks:', bev_metric_overlay_path)


In [ ]:
#@title 7. 3D preview with metric cuboids
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def cuboid_vertices(cx, cy, cz, length, width, height, yaw):
    x = length / 2.0
    y = width / 2.0
    z = height / 2.0
    local = np.array([
        [-x, -y, -z], [ x, -y, -z], [ x,  y, -z], [-x,  y, -z],
        [-x, -y,  z], [ x, -y,  z], [ x,  y,  z], [-x,  y,  z],
    ], dtype=float)
    c = math.cos(yaw)
    s = math.sin(yaw)
    R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=float)
    return local @ R.T + np.array([cx, cy, cz])

def cuboid_faces(verts):
    return [
        [verts[i] for i in [0, 1, 2, 3]],
        [verts[i] for i in [4, 5, 6, 7]],
        [verts[i] for i in [0, 1, 5, 4]],
        [verts[i] for i in [1, 2, 6, 5]],
        [verts[i] for i in [2, 3, 7, 6]],
        [verts[i] for i in [3, 0, 4, 7]],
    ]

PREVIEW_FRAME_ID = int(cuboids_df['frame_id'].min()) #@param {type:'integer'}
frame_df = cuboids_df[cuboids_df['frame_id'] == PREVIEW_FRAME_ID].copy()
if len(frame_df) == 0:
    PREVIEW_FRAME_ID = int(cuboids_df['frame_id'].iloc[0])
    frame_df = cuboids_df[cuboids_df['frame_id'] == PREVIEW_FRAME_ID].copy()

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

road_w_m = BEV_WIDTH * meters_per_pixel
road_h_m = BEV_HEIGHT * meters_per_pixel
ax.plot([0, road_w_m, road_w_m, 0, 0], [0, 0, road_h_m, road_h_m, 0], [0, 0, 0, 0, 0], linewidth=2)

for _, row in frame_df.iterrows():
    verts = cuboid_vertices(row['center_x_m'], row['center_y_m'], row['center_z_m'], row['length_m'], row['width_m'], row['height_m'], row['yaw_rad'])
    faces = cuboid_faces(verts)
    poly = Poly3DCollection(faces, alpha=0.35, edgecolor='k')
    ax.add_collection3d(poly)
    ax.text(row['center_x_m'], row['center_y_m'], row['height_m'] + 0.4, str(int(row['track_id'])), fontsize=8)

ax.set_title('Metric 3D cuboids on road plane, frame ' + str(PREVIEW_FRAME_ID))
ax.set_xlabel('X meters')
ax.set_ylabel('Y meters')
ax.set_zlabel('Z meters')
ax.set_xlim(0, road_w_m)
ax.set_ylim(0, road_h_m)
ax.set_zlim(0, 8)
ax.set_box_aspect((road_w_m, road_h_m, 8 * 3))
ax.view_init(elev=35, azim=-60)

preview_3d_path = NB04_DIR / 'metric_3d_cuboids_preview.png'
plt.savefig(preview_3d_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved 3D preview:', preview_3d_path)


In [ ]:
#@title 8. Export simple OBJ scene for one frame
OBJ_FRAME_ID = PREVIEW_FRAME_ID #@param {type:'integer'}
obj_df = cuboids_df[cuboids_df['frame_id'] == OBJ_FRAME_ID].copy()
if len(obj_df) == 0:
    raise ValueError('No cuboids for OBJ_FRAME_ID: ' + str(OBJ_FRAME_ID))

obj_path = NB04_DIR / ('scene_frame_' + str(OBJ_FRAME_ID).zfill(6) + '.obj')

vertices = []
faces = []

road_w_m = BEV_WIDTH * meters_per_pixel
road_h_m = BEV_HEIGHT * meters_per_pixel
road_start = len(vertices) + 1
vertices.extend([
    (0, 0, 0),
    (road_w_m, 0, 0),
    (road_w_m, road_h_m, 0),
    (0, road_h_m, 0),
])
faces.append((road_start, road_start + 1, road_start + 2, road_start + 3))

for _, row in obj_df.iterrows():
    verts = cuboid_vertices(row['center_x_m'], row['center_y_m'], row['center_z_m'], row['length_m'], row['width_m'], row['height_m'], row['yaw_rad'])
    start = len(vertices) + 1
    vertices.extend([tuple(v) for v in verts])
    for face_ids in [[0,1,2,3], [4,5,6,7], [0,1,5,4], [1,2,6,5], [2,3,7,6], [3,0,4,7]]:
        faces.append(tuple(start + i for i in face_ids))

with open(obj_path, 'w') as f:
    f.write('# UAVDT metric cuboid scene export\n')
    f.write('# Frame ' + str(OBJ_FRAME_ID) + '\n')
    for v in vertices:
        f.write('v {:.6f} {:.6f} {:.6f}\n'.format(v[0], v[1], v[2]))
    for face in faces:
        f.write('f ' + ' '.join(str(i) for i in face) + '\n')

print('Saved OBJ scene:', obj_path)
print('Vertices:', len(vertices))
print('Faces:', len(faces))


In [ ]:
#@title 9. Export dynamic scene JSON
scene_json = {
    'coordinate_system': {
        'x': 'right in BEV, meters',
        'y': 'forward/up in BEV, meters',
        'z': 'up, meters',
    },
    'calibration': calibration_json,
    'source_files': {
        'homography_json': str(homography_path),
        'tracks_csv': str(tracks_path),
    },
    'vehicle_dimensions_default_m': {
        k: {'length': v[0], 'width': v[1], 'height': v[2]} for k, v in DEFAULT_DIMS.items()
    },
    'frames': [],
}

for frame_id, group in cuboids_df.groupby('frame_id'):
    frame_record = {
        'frame_id': int(frame_id),
        'time_s': float(frame_id / FPS),
        'vehicles': [],
    }
    for _, row in group.iterrows():
        frame_record['vehicles'].append({
            'track_id': int(row['track_id']),
            'class_name': row['class_name'],
            'center_m': [float(row['center_x_m']), float(row['center_y_m']), float(row['center_z_m'])],
            'size_m': [float(row['length_m']), float(row['width_m']), float(row['height_m'])],
            'yaw_rad': float(row['yaw_rad']),
            'speed_mps': float(row['speed_mps']),
        })
    scene_json['frames'].append(frame_record)

scene_json_path = NB04_DIR / 'dynamic_metric_scene.json'
with open(scene_json_path, 'w') as f:
    json.dump(scene_json, f, indent=2)

print('Saved dynamic scene JSON:', scene_json_path)
print('Number of frames:', len(scene_json['frames']))


## Next notebook suggestion

Notebook 05 can create an interactive renderer for the exported dynamic scene:

- load `dynamic_metric_scene.json`
- render the road plane and vehicles in Plotly or Three.js
- animate vehicle cuboids over time
- optionally replace cuboids with simple CAD car meshes.
